# 🎓 Plataforma Colaborativa de Estudos

Este notebook está dividido em células pequenas — rode **uma de cada vez**, na ordem, com `Shift+Enter`. Se alguma der erro, você sabe exatamente qual parte falhou.

**Antes de começar:** crie um arquivo `.env` na raiz do repositório com:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=exercicios_colaborativos
DB_USER=postgres
DB_PASSWORD=sua_senha
```

**Ordem de execução:**
1. Instalar bibliotecas
2. Imports
3. Conexão com o banco
4. CSS visual
5. Funções auxiliares
6. Tela de Usuários
7. Tela de Perguntas
8. Tela de Dashboard
9. Montar e abrir no navegador

Para parar o servidor depois: rode `_PLAT_SERVER.stop()` em uma nova célula.

## 2. Imports e configuração inicial

In [1]:
!pip install pandas sqlalchemy psycopg2-binary panel python-dotenv jupyter_bokeh

  Using cached jupyter_bokeh-4.1.0-py3-none-any.whl.metadata (7.2 kB)
Using cached jupyter_bokeh-4.1.0-py3-none-any.whl (1.3 MB)


In [2]:
# ===================================================
# PLATAFORMA COLABORATIVA DE ESTUDOS — CRUD (tela unica)
# ===================================================

import os
from dotenv import load_dotenv, find_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn

# Carrega as variaveis do arquivo .env (procura subindo nas pastas pai)
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print(f"[.env carregado de: {env_path}]")

[.env carregado de: C:\Users\grazy\Downloads\trabalho-fbd\.env]


In [3]:
# Le as variaveis de ambiente
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

if not all([DB_HOST, DB_NAME, DB_USER, DB_PASS]):
    raise RuntimeError(
        "\n\nARQUIVO .env NAO ENCONTRADO OU INCOMPLETO!\n"
        "Crie um .env na raiz do repositorio com:\n"
        "  DB_HOST=localhost\n"
        "  DB_PORT=5432\n"
        "  DB_NAME=exercicios_colaborativos\n"
        "  DB_USER=postgres\n"
        "  DB_PASSWORD=sua_senha\n"
    )

# Conexao psycopg2 (para INSERT/UPDATE/DELETE)
con = pg.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

# Engine SQLAlchemy (para SELECT com pandas)
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(cnx)
print("Conexao OK")

Conexao OK


In [4]:
# Inicializa as extensoes do Panel
pn.extension('tabulator', notifications=True)

pn.config.raw_css = ["""
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
body { font-family:'Inter',sans-serif !important; background:#F8FAFC !important; }
input, textarea, select, .bk-input, .bk-spin-wrapper input {
    color:#0F172A !important; background:#FFFFFF !important;
    border:1.5px solid #CBD5E1 !important; border-radius:7px !important;
    font-family:'Inter',sans-serif !important; font-size:13px !important;
}
input::placeholder, textarea::placeholder { color:#94A3B8 !important; opacity:1 !important; }
.bk label, label.bk, div.bk label, .bk-clearfix label, .bk-widget label {
    color:#1E293B !important; font-size:12px !important;
    font-weight:700 !important; font-family:'Inter',sans-serif !important;
}
.bk-btn {
    font-family:'Inter',sans-serif !important; font-size:13px !important;
    font-weight:700 !important; border-radius:7px !important; padding:7px 16px !important;
}
.bk-btn-success { background:#10B981 !important; color:#fff !important; border:none !important; }
.bk-btn-primary { background:#4F46E5 !important; color:#fff !important; border:none !important; }
.bk-btn-danger  { background:#F43F5E !important; color:#fff !important; border:none !important; }
.bk-btn-default { background:#F1F5F9 !important; color:#334155 !important; border:1.5px solid #CBD5E1 !important; }
.bk-btn:hover   { filter:brightness(.93) !important; }
.tabulator { border-radius:10px !important; overflow:hidden !important; font-family:'Inter',sans-serif !important; }
.tabulator-header, .tabulator-col, .tabulator-col-content, .tabulator-col-title {
    background:#1E293B !important; color:#FFFFFF !important;
    font-size:11px !important; font-weight:700 !important;
    text-transform:uppercase !important; letter-spacing:.06em !important;
}
.tabulator-row:nth-child(even) { background:#F8FAFC !important; }
.tabulator-row:hover { background:#EEF2FF !important; }
.tabulator-cell { color:#0F172A !important; font-size:13px !important; }
"""]

NotificationArea()

In [18]:
# ===================================================
# WIDGETS — campos de entrada (USUARIOS)
# ===================================================

flag = ''  # usado para ignorar filtros em branco nas consultas

nome = pn.widgets.TextInput(name="Nome *", placeholder="Ex: Ana", sizing_mode="stretch_width")
sobrenome = pn.widgets.TextInput(name="Sobrenome *", placeholder="Ex: Silva", sizing_mode="stretch_width")
email = pn.widgets.TextInput(name="E-mail *", placeholder="ana@email.com", sizing_mode="stretch_width")
senha = pn.widgets.PasswordInput(name="Senha *", placeholder="Senha", sizing_mode="stretch_width")
foto = pn.widgets.TextInput(name="Foto de perfil (URL, opcional)", placeholder="https://...", sizing_mode="stretch_width")
biografia = pn.widgets.TextAreaInput(name="Biografia", placeholder="(opcional)", height=60, sizing_mode="stretch_width")
modo_focado = pn.widgets.Checkbox(name="Modo Focado")

tipo_perfil = pn.widgets.Select(
    name="Tipo de Perfil *",
    options={"Estudante": "estudante", "Colaborador": "colaborador", "Estudante + Colaborador": "ambos"},
    sizing_mode="stretch_width"
)
curso = pn.widgets.TextInput(name="Curso (se Estudante)", placeholder="Ex: Ciencia da Computacao", sizing_mode="stretch_width")

filtro_busca = pn.widgets.TextInput(name="Buscar (nome, sobrenome ou email)", placeholder="Digite para filtrar...", sizing_mode="stretch_width")
id_usuario_alvo = pn.widgets.IntInput(name="ID do Usuario (para Atualizar/Excluir)", value=0, start=0, sizing_mode="stretch_width")

In [11]:
# ===================================================
# BOTOES — USUARIOS
# ===================================================

buttonConsultar = pn.widgets.Button(name='Consultar', button_type='primary')
buttonInserir   = pn.widgets.Button(name='Inserir',   button_type='success')
buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='primary')
buttonExcluir   = pn.widgets.Button(name='Excluir',   button_type='danger')
buttonLimpar    = pn.widgets.Button(name='Limpar',    button_type='default')

In [19]:
# ===================================================
# FUNCOES CRUD — USUARIOS
# ===================================================

def queryAll():
    """Consulta todos os usuarios, com tipo de perfil (Estudante/Colaborador) calculado via JOIN."""
    query = """
        SELECT u.id_usuario AS "ID", u.p_nome AS "Nome", u.sobrenome AS "Sobrenome",
               u.email AS "Email", u.modo_focado AS "Focado",
               CASE WHEN u.foto IS NOT NULL AND u.foto != '' THEN 'Sim' ELSE 'Nao' END AS "Foto",
               CASE WHEN e.id_usuario IS NOT NULL AND c.id_usuario IS NOT NULL THEN 'Est+Col'
                    WHEN e.id_usuario IS NOT NULL THEN 'Estudante'
                    WHEN c.id_usuario IS NOT NULL THEN 'Colaborador'
                    ELSE 'Sem perfil' END AS "Tipo",
               COALESCE(e.curso,'') AS "Curso"
        FROM Usuario u
        LEFT JOIN Estudante e ON u.id_usuario = e.id_usuario
        LEFT JOIN Colaborador c ON u.id_usuario = c.id_usuario
        ORDER BY u.id_usuario
    """
    df = pd.read_sql_query(query, cnx)
    return pn.widgets.Tabulator(
        df, height=320, pagination='local', page_size=8,
        sizing_mode='stretch_width', layout='fit_data_stretch',
        theme='site', show_index=False
    )


def on_consultar():
    """Consulta usuarios filtrando por nome, sobrenome ou email. Se vazio, retorna todos."""
    try:
        termo = filtro_busca.value_input.strip()
        query = """
            SELECT u.id_usuario AS "ID", u.p_nome AS "Nome", u.sobrenome AS "Sobrenome",
                   u.email AS "Email", u.modo_focado AS "Focado",
                   CASE WHEN u.foto IS NOT NULL AND u.foto != '' THEN 'Sim' ELSE 'Nao' END AS "Foto",
                   CASE WHEN e.id_usuario IS NOT NULL AND c.id_usuario IS NOT NULL THEN 'Est+Col'
                        WHEN e.id_usuario IS NOT NULL THEN 'Estudante'
                        WHEN c.id_usuario IS NOT NULL THEN 'Colaborador'
                        ELSE 'Sem perfil' END AS "Tipo",
                   COALESCE(e.curso,'') AS "Curso"
            FROM Usuario u
            LEFT JOIN Estudante e ON u.id_usuario = e.id_usuario
            LEFT JOIN Colaborador c ON u.id_usuario = c.id_usuario
            WHERE (%(termo)s = %(flag)s
                   OR u.p_nome ILIKE %(like)s
                   OR u.sobrenome ILIKE %(like)s
                   OR u.email ILIKE %(like)s)
            ORDER BY u.id_usuario
        """
        df = pd.read_sql_query(query, cnx, params={"termo": termo, "flag": flag, "like": f"%{termo}%"})
        table = pn.widgets.Tabulator(
            df, height=320, pagination='local', page_size=8,
            sizing_mode='stretch_width', layout='fit_data_stretch',
            theme='site', show_index=False
        )
        return table
    except Exception as e:
        return pn.pane.Alert(f'Nao foi possivel consultar: {str(e)}', alert_type='danger')


def on_inserir():
    """Insere um novo Usuario e, conforme o tipo escolhido, em Estudante e/ou Colaborador."""
    cursor = con.cursor()
    try:
        if not all([nome.value_input.strip(), sobrenome.value_input.strip(),
                    email.value_input.strip(), senha.value]):
            return pn.pane.Alert('Preencha nome, sobrenome, e-mail e senha.', alert_type='warning')

        cursor.execute(
            "INSERT INTO Usuario (p_nome, sobrenome, email, senha, modo_focado, biografia, foto) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id_usuario",
            (nome.value_input.strip(), sobrenome.value_input.strip(), email.value_input.strip(),
             senha.value, modo_focado.value, biografia.value.strip() or None,
             foto.value_input.strip() or None)
        )
        novo_id = cursor.fetchone()[0]

        tipo = tipo_perfil.value
        if tipo in ('estudante', 'ambos'):
            cursor.execute("INSERT INTO Estudante (id_usuario, curso) VALUES (%s, %s)",
                            (novo_id, curso.value_input.strip() or None))
        if tipo in ('colaborador', 'ambos'):
            cursor.execute("INSERT INTO Colaborador (id_usuario) VALUES (%s)", (novo_id,))

        con.commit()
        cursor.close()
        pn.state.notifications.success(f'Usuario #{novo_id} inserido com sucesso!')
        return queryAll()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel inserir: {str(e)}', alert_type='danger')


def on_atualizar():
    """Atualiza os dados do Usuario com o ID informado em id_usuario_alvo."""
    cursor = con.cursor()
    try:
        if not id_usuario_alvo.value:
            return pn.pane.Alert('Informe o ID do usuario que deseja atualizar.', alert_type='warning')

        cursor.execute(
            "UPDATE Usuario SET p_nome=%s, sobrenome=%s, email=%s, modo_focado=%s, "
            "biografia=%s, foto=%s WHERE id_usuario=%s",
            (nome.value_input.strip(), sobrenome.value_input.strip(), email.value_input.strip(),
             modo_focado.value, biografia.value.strip() or None,
             foto.value_input.strip() or None, id_usuario_alvo.value)
        )
        if curso.value_input.strip():
            cursor.execute("UPDATE Estudante SET curso=%s WHERE id_usuario=%s",
                            (curso.value_input.strip(), id_usuario_alvo.value))
        con.commit()
        cursor.close()
        pn.state.notifications.success('Usuario atualizado com sucesso!')
        return queryAll()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel atualizar: {str(e)}', alert_type='danger')


def on_excluir():
    """Exclui o Usuario com o ID informado em id_usuario_alvo (cascade remove Estudante/Colaborador)."""
    cursor = con.cursor()
    try:
        if not id_usuario_alvo.value:
            return pn.pane.Alert('Informe o ID do usuario que deseja excluir.', alert_type='warning')

        cursor.execute("DELETE FROM Usuario WHERE id_usuario=%s", (id_usuario_alvo.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success('Usuario excluido com sucesso!')
        return queryAll()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel excluir: {str(e)}', alert_type='danger')


def on_limpar():
    """Limpa todos os campos do formulario."""
    nome.value = ''
    sobrenome.value = ''
    email.value = ''
    senha.value = ''
    foto.value = ''
    biografia.value = ''
    curso.value = ''
    modo_focado.value = False
    tipo_perfil.value = 'estudante'
    id_usuario_alvo.value = 0
    return queryAll()

In [14]:
# ===================================================
# WIDGETS — PERGUNTAS
# ===================================================

_df_disc = pd.read_sql_query("SELECT id_disciplina, nome FROM Disciplina ORDER BY nome", cnx)
_disc_options = {row['nome']: int(row['id_disciplina']) for _, row in _df_disc.iterrows()} or {"(sem disciplinas)": None}

_df_est = pd.read_sql_query("""
    SELECT e.id_usuario, u.p_nome, u.sobrenome
    FROM Estudante e JOIN Usuario u ON e.id_usuario = u.id_usuario
    ORDER BY e.id_usuario
""", cnx)
_estudante_options = {"— Selecione —": None}
for _, r in _df_est.iterrows():
    _estudante_options[f"{r['p_nome']} {r['sobrenome']} (#{int(r['id_usuario'])})"] = int(r['id_usuario'])

titulo_pergunta = pn.widgets.TextInput(name="Titulo *", placeholder="Resumo da duvida", sizing_mode="stretch_width")
corpo_pergunta  = pn.widgets.TextAreaInput(name="Descricao *", placeholder="Descreva a pergunta em detalhes...", height=80, sizing_mode="stretch_width")
disciplina_pergunta = pn.widgets.Select(name="Disciplina *", options=_disc_options, sizing_mode="stretch_width")
estudante_pergunta  = pn.widgets.Select(name="Estudante *", options=_estudante_options, sizing_mode="stretch_width")
resolvido_pergunta  = pn.widgets.Checkbox(name="Marcar como resolvida")
anexo_pergunta      = pn.widgets.TextInput(name="URL da imagem (opcional)", placeholder="https://...", sizing_mode="stretch_width")

filtro_busca_pergunta = pn.widgets.TextInput(name="Buscar por titulo", placeholder="Digite para filtrar...", sizing_mode="stretch_width")
id_pergunta_alvo = pn.widgets.IntInput(name="ID da Pergunta (para Atualizar/Excluir)", value=0, start=0, sizing_mode="stretch_width")

In [15]:
# ===================================================
# BOTOES — PERGUNTAS
# ===================================================

buttonConsultarPergunta = pn.widgets.Button(name='Consultar', button_type='primary')
buttonPublicarPergunta  = pn.widgets.Button(name='Publicar',  button_type='success')
buttonAtualizarPergunta = pn.widgets.Button(name='Atualizar Status', button_type='primary')
buttonExcluirPergunta   = pn.widgets.Button(name='Excluir',   button_type='danger')
buttonLimparPergunta    = pn.widgets.Button(name='Limpar',    button_type='default')

In [16]:
# ===================================================
# FUNCOES CRUD — PERGUNTAS
# ===================================================

def queryAllPerguntas():
    """Consulta todas as perguntas, com disciplina e autor via JOIN."""
    query = """
        SELECT p.id_conteudo AS "ID", p.titulo_resumido AS "Titulo",
               LEFT(c.corpo_do_texto, 60) AS "Resumo",
               d.nome AS "Disciplina", p.status_resolvido AS "Resolvido",
               u.p_nome||' '||u.sobrenome AS "Autor",
               TO_CHAR(c.data_criacao,'DD/MM/YYYY') AS "Data"
        FROM Pergunta p
        JOIN Conteudo c ON p.id_conteudo = c.id_conteudo
        JOIN Disciplina d ON p.id_disciplina = d.id_disciplina
        JOIN Estudante est ON p.id_estudante_criador = est.id_usuario
        JOIN Usuario u ON est.id_usuario = u.id_usuario
        ORDER BY c.data_criacao DESC
        LIMIT 60
    """
    df = pd.read_sql_query(query, cnx)
    return pn.widgets.Tabulator(
        df, height=320, pagination='local', page_size=8,
        sizing_mode='stretch_width', layout='fit_data_stretch',
        theme='site', show_index=False
    )


def on_consultar_pergunta():
    """Consulta perguntas filtrando pelo titulo. Se vazio, retorna todas."""
    try:
        termo = filtro_busca_pergunta.value_input.strip()
        query = """
            SELECT p.id_conteudo AS "ID", p.titulo_resumido AS "Titulo",
                   LEFT(c.corpo_do_texto, 60) AS "Resumo",
                   d.nome AS "Disciplina", p.status_resolvido AS "Resolvido",
                   u.p_nome||' '||u.sobrenome AS "Autor",
                   TO_CHAR(c.data_criacao,'DD/MM/YYYY') AS "Data"
            FROM Pergunta p
            JOIN Conteudo c ON p.id_conteudo = c.id_conteudo
            JOIN Disciplina d ON p.id_disciplina = d.id_disciplina
            JOIN Estudante est ON p.id_estudante_criador = est.id_usuario
            JOIN Usuario u ON est.id_usuario = u.id_usuario
            WHERE (%(termo)s = %(flag)s OR p.titulo_resumido ILIKE %(like)s)
            ORDER BY c.data_criacao DESC
            LIMIT 60
        """
        df = pd.read_sql_query(query, cnx, params={"termo": termo, "flag": flag, "like": f"%{termo}%"})
        return pn.widgets.Tabulator(
            df, height=320, pagination='local', page_size=8,
            sizing_mode='stretch_width', layout='fit_data_stretch',
            theme='site', show_index=False
        )
    except Exception as e:
        return pn.pane.Alert(f'Nao foi possivel consultar: {str(e)}', alert_type='danger')


def on_publicar_pergunta():
    """Insere uma nova Pergunta (cria o Conteudo base e depois a Pergunta)."""
    cursor = con.cursor()
    try:
        if not all([titulo_pergunta.value_input.strip(), corpo_pergunta.value.strip(),
                    disciplina_pergunta.value, estudante_pergunta.value]):
            return pn.pane.Alert('Preencha titulo, descricao, disciplina e selecione um estudante.', alert_type='warning')

        cursor.execute(
            "INSERT INTO Conteudo (corpo_do_texto, data_criacao, id_autor) "
            "VALUES (%s, NOW(), %s) RETURNING id_conteudo",
            (corpo_pergunta.value.strip(), estudante_pergunta.value)
        )
        id_conteudo = cursor.fetchone()[0]

        cursor.execute(
            "INSERT INTO Pergunta (id_conteudo, titulo_resumido, anexo_imagem, "
            "id_disciplina, id_estudante_criador, status_resolvido) "
            "VALUES (%s, %s, %s, %s, %s, %s)",
            (id_conteudo, titulo_pergunta.value_input.strip(),
             anexo_pergunta.value_input.strip() or None,
             disciplina_pergunta.value, estudante_pergunta.value, resolvido_pergunta.value)
        )
        con.commit()
        cursor.close()
        pn.state.notifications.success(f'Pergunta #{id_conteudo} publicada com sucesso!')
        return queryAllPerguntas()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel publicar: {str(e)}', alert_type='danger')


def on_atualizar_pergunta():
    """Atualiza o status_resolvido da Pergunta com o ID informado."""
    cursor = con.cursor()
    try:
        if not id_pergunta_alvo.value:
            return pn.pane.Alert('Informe o ID da pergunta que deseja atualizar.', alert_type='warning')

        cursor.execute(
            "UPDATE Pergunta SET status_resolvido=%s WHERE id_conteudo=%s",
            (resolvido_pergunta.value, id_pergunta_alvo.value)
        )
        con.commit()
        cursor.close()
        pn.state.notifications.success('Status da pergunta atualizado!')
        return queryAllPerguntas()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel atualizar: {str(e)}', alert_type='danger')


def on_excluir_pergunta():
    """Exclui a Pergunta (e o Conteudo associado, via cascade) com o ID informado."""
    cursor = con.cursor()
    try:
        if not id_pergunta_alvo.value:
            return pn.pane.Alert('Informe o ID da pergunta que deseja excluir.', alert_type='warning')

        cursor.execute("DELETE FROM Conteudo WHERE id_conteudo=%s", (id_pergunta_alvo.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success('Pergunta excluida com sucesso!')
        return queryAllPerguntas()
    except Exception as e:
        con.rollback()
        cursor.close()
        return pn.pane.Alert(f'Nao foi possivel excluir: {str(e)}', alert_type='danger')


def on_limpar_pergunta():
    """Limpa todos os campos do formulario de perguntas."""
    titulo_pergunta.value = ''
    corpo_pergunta.value = ''
    anexo_pergunta.value = ''
    estudante_pergunta.value = None
    resolvido_pergunta.value = False
    id_pergunta_alvo.value = 0
    return queryAllPerguntas()

In [20]:
# ===================================================
# LIGACAO BOTOES <-> FUNCOES + LAYOUT FINAL
# ===================================================

def table_creator(cons, ins, atu, exc, limp):
    if cons: return on_consultar()
    if ins:  return on_inserir()
    if atu:  return on_atualizar()
    if exc:  return on_excluir()
    if limp: return on_limpar()
    return queryAll()

interactive_table = pn.bind(
    table_creator,
    buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir, buttonLimpar
)

def table_creator_pergunta(cons, pub, atu, exc, limp):
    if cons: return on_consultar_pergunta()
    if pub:  return on_publicar_pergunta()
    if atu:  return on_atualizar_pergunta()
    if exc:  return on_excluir_pergunta()
    if limp: return on_limpar_pergunta()
    return queryAllPerguntas()

interactive_table_pergunta = pn.bind(
    table_creator_pergunta,
    buttonConsultarPergunta, buttonPublicarPergunta,
    buttonAtualizarPergunta, buttonExcluirPergunta, buttonLimparPergunta
)

header = pn.pane.HTML("""
<div style="background:linear-gradient(135deg,#4F46E5,#7C3AED);
            border-radius:12px;padding:20px 26px;margin-bottom:16px;">
  <div style="display:flex;align-items:center;gap:12px;">
    <span style="font-size:28px;">&#127891;</span>
    <div>
      <div style="font-family:'DM Sans',sans-serif;font-size:19px;font-weight:700;color:#fff;">
        Plataforma Colaborativa de Estudos — CRUD</div>
      <div style="font-size:12px;color:rgba(255,255,255,.65);margin-top:2px;">
        Consulte, insira, atualize e exclua usuarios e perguntas.</div>
    </div>
  </div>
</div>
""", sizing_mode="stretch_width")

botoes_usuario = pn.Column(
    pn.Row(buttonConsultar, buttonInserir, sizing_mode="stretch_width"),
    pn.Row(buttonAtualizar, buttonExcluir, sizing_mode="stretch_width"),
    pn.Row(buttonLimpar, sizing_mode="stretch_width"),
    sizing_mode="stretch_width",
)

formulario_usuario = pn.Column(
    pn.pane.HTML('<p style="font-size:13px;font-weight:700;color:#334155;margin:0 0 12px;'
                 'padding-bottom:8px;border-bottom:1.5px solid #F1F5F9;">Dados do Usuario</p>'),
    pn.Row(nome, sobrenome, sizing_mode="stretch_width"),
    pn.Row(email, senha, sizing_mode="stretch_width"),
    pn.Row(tipo_perfil, curso, sizing_mode="stretch_width"),
    foto,
    pn.Row(modo_focado),
    biografia,
    pn.layout.Divider(),
    filtro_busca,
    id_usuario_alvo,
    botoes_usuario,
    styles={"background": "#FFFFFF", "border-radius": "12px", "padding": "22px",
            "box-shadow": "0 1px 6px rgba(0,0,0,.08)"},
    sizing_mode="stretch_width",
    width=420,
    min_width=380,
)

secao_usuarios = pn.Column(
    pn.pane.HTML('<h2 style="font-size:18px;font-weight:700;color:#0F172A;margin:0 0 12px;">👥 Usuarios</h2>'),
    pn.Row(
        formulario_usuario,
        pn.Column(interactive_table, sizing_mode='stretch_width', min_width=500),
        sizing_mode='stretch_width',
    ),
    sizing_mode='stretch_width',
)

botoes_pergunta = pn.Column(
    pn.Row(buttonConsultarPergunta, buttonPublicarPergunta, sizing_mode="stretch_width"),
    pn.Row(buttonAtualizarPergunta, buttonExcluirPergunta, sizing_mode="stretch_width"),
    pn.Row(buttonLimparPergunta, sizing_mode="stretch_width"),
    sizing_mode="stretch_width",
)

aviso_pergunta = pn.pane.HTML(
    '<div style="background:#FFFBEB;border-left:3px solid #F59E0B;border-radius:7px;'
    'padding:8px 13px;font-size:12px;color:#92400E;margin-bottom:10px;">'
    '<b>Atencao:</b> So aparecem estudantes ja cadastrados na secao acima '
    '(usuario precisa ser do tipo Estudante).</div>'
)

formulario_pergunta = pn.Column(
    pn.pane.HTML('<p style="font-size:13px;font-weight:700;color:#334155;margin:0 0 12px;'
                 'padding-bottom:8px;border-bottom:1.5px solid #F1F5F9;">Dados da Pergunta</p>'),
    aviso_pergunta,
    titulo_pergunta,
    corpo_pergunta,
    pn.Row(disciplina_pergunta, estudante_pergunta, sizing_mode="stretch_width"),
    pn.Row(resolvido_pergunta),
    anexo_pergunta,
    pn.layout.Divider(),
    filtro_busca_pergunta,
    id_pergunta_alvo,
    botoes_pergunta,
    styles={"background": "#FFFFFF", "border-radius": "12px", "padding": "22px",
            "box-shadow": "0 1px 6px rgba(0,0,0,.08)"},
    sizing_mode="stretch_width",
    width=420,
    min_width=380,
)

secao_perguntas = pn.Column(
    pn.pane.HTML('<h2 style="font-size:18px;font-weight:700;color:#0F172A;margin:0 0 12px;">❓ Perguntas</h2>'),
    pn.Row(
        formulario_pergunta,
        pn.Column(interactive_table_pergunta, sizing_mode='stretch_width', min_width=500),
        sizing_mode='stretch_width',
    ),
    sizing_mode='stretch_width',
)

app = pn.Column(
    header,
    secao_usuarios,
    pn.layout.Divider(),
    secao_perguntas,
    sizing_mode='stretch_width',
    styles={"background": "#F8FAFC", "padding": "18px"},
)

try:
    if '_PLAT_SERVER' in globals() and _PLAT_SERVER is not None:
        _PLAT_SERVER.stop()
except Exception:
    pass

_PLAT_SERVER = app.show(threaded=True)
print("App aberto no navegador. Para parar: _PLAT_SERVER.stop()")

App aberto no navegador. Para parar: _PLAT_SERVER.stop()
Launching server at http://localhost:62333
